# Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans


In [0]:
processed_table = spark.table("aml_framework.aml_preprocessed_table")

# Log Transform highly skewed features

In [0]:
numeric_cols = [
    field.name
    for field in processed_table.schema.fields
    if isinstance(
        field.dataType,
        (IntegerType, LongType, FloatType, DoubleType, DecimalType, ShortType)
    )
]

skewness_exprs = [
    F.skewness(F.col(c)).alias(c)
    for c in numeric_cols
]

skewness_df = processed_table.select(*skewness_exprs)

skewness_df.display()

In [0]:
skewness_row = skewness_df.collect()[0].asDict()

skewed_cols = [
    col
    for col, skew in skewness_row.items()
    if skew is not None and abs(skew) > 2
]

print("Highly skewed columns:")
print(skewed_cols)

In [0]:
processed_table_log = processed_table_log
for col in skewed_cols:
    min_val = processed_table.agg(F.min(col)).collect()[0][0]

    if min_val >= 0:
        # print(col)
        processed_table_log = processed_table_log.withColumn(
            f"{col}_log",
            F.log1p(F.col(col))
        )

processed_table_log.display()

# Assemble Features

In [0]:
feature_cols = numeric_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml = assembler.transform(processed_table_log)
# df_ml.display()

In [0]:
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(df_ml)

df_ml = scaler_model.transform(df_ml)
# df_ml.display()

# K-Means Clustering

In [0]:
kmeans = KMeans(
    k=100,
    featuresCol="scaled_features",
    predictionCol="cluster"
)

model = kmeans.fit(df_ml)

predictions = model.transform(df_ml)
predictions.display()

# Distance from center

In [0]:
centers = model.clusterCenters()

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
import numpy as np

def euclidean_distance(v, cluster):
    center = centers[cluster]
    return float(np.linalg.norm(np.array(v) - np.array(center)))

distance_udf = udf(euclidean_distance, DoubleType())

predictions = predictions.withColumn(
    "distance_from_center",
    distance_udf("scaled_features", "cluster")
)

predictions.display()

# Setting Threshold

In [0]:
threshold = predictions.approxQuantile(
    "distance_from_center",
    [0.85],
    0
)[0]
print("Threshold: ", threshold)
alerts_pred = predictions.withColumn(
    "predicted_fraud",
    F.when(F.col("distance_from_center") >= threshold, 1).otherwise(0)
).withColumn("IS_FRAUD", F.when(F.col("IS_FRAUD") == 'true', 1).otherwise(0))

alerts_pred.display()

# Confusion matrix

In [0]:
confusion_matrix = (
    alerts_pred
    .groupBy("IS_FRAUD", "predicted_fraud")
    .count()
)

confusion_matrix.display()

In [0]:
cm = (
    alerts_pred
    .groupBy("IS_FRAUD", "predicted_fraud")
    .count()
    .collect()
)

TP = TN = FP = FN = 0

for row in cm:

    actual = row["IS_FRAUD"]
    pred = row["predicted_fraud"]
    cnt = row["count"]

    if actual == 1 and pred == 1:
        TP = cnt

    elif actual == 0 and pred == 0:
        TN = cnt

    elif actual == 0 and pred == 1:
        FP = cnt

    elif actual == 1 and pred == 0:
        FN = cnt

print(f"TP={TP}, TN={TN}, FP={FP}, FN={FN}")

# Precision

In [0]:
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
precision

# Recall

In [0]:
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
recall

# Accuracy

In [0]:
accuracy = (TP + TN) / (TP + TN + FP + FN)
accuracy

# F1 score

In [0]:
f1 = (
    2 * precision * recall /
    (precision + recall)
    if (precision + recall) > 0
    else 0
)
f1

In [0]:
processed_table.filter("IS_FRAUD == 'true'").select("ACCOUNT_ID").distinct().count()/processed_table.select("ACCOUNT_ID").distinct().count()

In [0]:
processed_table.filter("IS_FRAUD == 'true'").display()